In [1]:
import torch
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm


In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)
print("model:", model_name)


device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model: sentence-transformers/all-MiniLM-L6-v2


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
batch_size = 128
emb1_batches = []
emb2_batches = []

for i in tqdm(range(0, len(ds), batch_size), desc="Encoding"):
    batch_s1 = sent1[i:i + batch_size]
    batch_s2 = sent2[i:i + batch_size]

    e1 = model.encode(
        batch_s1,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    e2 = model.encode(
        batch_s2,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    emb1_batches.append(e1)
    emb2_batches.append(e2)

emb1 = torch.cat(emb1_batches, dim=0)
emb2 = torch.cat(emb2_batches, dim=0)

print("emb1 shape:", tuple(emb1.shape))
print("emb2 shape:", tuple(emb2.shape))


Encoding:   0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 384)
emb2 shape: (408, 384)


In [5]:
threshold = 0.80

similarities = (emb1 * emb2).sum(dim=1)
scores = similarities.detach().cpu().numpy()
y_pred = (scores >= threshold).astype(np.int64)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1, "threshold": threshold})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6764705882352942, 'f1': 0.7441860465116279, 'threshold': 0.8}
                precision    recall  f1-score   support

not_paraphrase       0.49      0.65      0.56       129
    paraphrase       0.81      0.69      0.74       279

      accuracy                           0.68       408
     macro avg       0.65      0.67      0.65       408
  weighted avg       0.71      0.68      0.69       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9382226467132568
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.4680331349372864
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.904844343662262
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will deci

In [7]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": device,
    "threshold": float(threshold),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'sentence-transformers/all-MiniLM-L6-v2',
 'device': 'mps',
 'threshold': 0.8,
 'num_examples': 408,
 'accuracy': 0.6764705882352942,
 'f1': 0.7441860465116279}